# 03 — Hypothesis Testing & Comparative Analysis

Based on patterns observed in EDA (notebook 02), test specific questions:

1. What is driving the admission growth? (Procedure decomposition)
2. Is there a geographic access gap? (Patient migration)
3. Why do some hospitals treat patients faster than others? (Matched comparison)
4. What hospital-level factors correlate with shorter stays?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from scipy import stats

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)

OUTPUT_DIR = Path("../outputs")
PLOT_DIR = OUTPUT_DIR / "plots"
METRICS_DIR = OUTPUT_DIR / "metrics"

kidney = pd.read_parquet(OUTPUT_DIR / "kidney_sih.parquet")
print(f"Loaded {len(kidney):,} records")

# Prepare common columns (SEXO and CAR_INT are strings in parquet)
kidney["PROC_REA"] = kidney["PROC_REA"].astype(str).str.strip()
kidney["sex_label"] = kidney["SEXO"].astype(str).map({"1": "Male", "3": "Female"}).fillna("Unknown")
kidney["admission_type"] = kidney["CAR_INT"].astype(str).map({"01": "Elective", "02": "Emergency"}).fillna("Other")
kidney["is_emergency"] = (kidney["CAR_INT"].astype(str) == "02").astype(int)
kidney["migrated"] = kidney["MUNIC_RES"].astype(str) != kidney["MUNIC_MOV"].astype(str)

## 1. Procedure Decomposition

How much of the admission growth comes from specific procedure codes?

In [ ]:
# Get top procedures and their yearly counts
top_procs = kidney["PROC_REA"].value_counts().head(8).index.tolist()

proc_yearly = kidney.groupby(["year", "PROC_REA"]).size().unstack(fill_value=0)

# Growth contribution by procedure
first_year = proc_yearly.iloc[0]
last_year = proc_yearly.iloc[-1]
growth_by_proc = (last_year - first_year).sort_values(ascending=False)
total_growth = growth_by_proc.sum()

print(f"Total growth: {total_growth:,.0f} admissions")
print(f"\nGrowth contribution by procedure (top 10):")
for proc, g in growth_by_proc.head(10).items():
    print(f"  {proc}: +{g:,.0f} ({g/total_growth*100:.1f}% of total growth)")

In [ ]:
# Visualize growth contributors
top_growth = growth_by_proc.head(8)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ["#4CAF50" if v > 0 else "#F44336" for v in top_growth.values]
axes[0].barh(range(len(top_growth)), top_growth.values, color=colors)
axes[0].set_yticks(range(len(top_growth)))
axes[0].set_yticklabels(top_growth.index, fontsize=8)
axes[0].set_title("Growth Contribution by Procedure")
axes[0].set_xlabel("Change in Admissions (first to last year)")
axes[0].invert_yaxis()

# Track top 5 procedures over time
for proc in top_procs[:5]:
    if proc in proc_yearly.columns:
        axes[1].plot(proc_yearly.index, proc_yearly[proc], marker="o", label=proc[:15])
axes[1].set_title("Top Procedures Over Time")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Admissions")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(PLOT_DIR / "03_procedure_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Procedure × Length of Stay

Do different procedures have different lengths of stay?

In [ ]:
# Stats by procedure
proc_stats = kidney[kidney["PROC_REA"].isin(top_procs)].groupby("PROC_REA").agg(
    count=("DIAS_PERM", "count"),
    mean_stay=("DIAS_PERM", "mean"),
    median_stay=("DIAS_PERM", "median"),
    mean_cost=("VAL_TOT", "mean"),
    mortality=("MORTE", "mean"),
    pct_emergency=("is_emergency", "mean"),
).sort_values("count", ascending=False)

print(proc_stats.round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 5))
subset = kidney[kidney["PROC_REA"].isin(top_procs[:6]) & (kidney["DIAS_PERM"] <= 15)]
sns.boxplot(data=subset, x="PROC_REA", y="DIAS_PERM", ax=ax)
ax.set_title("Length of Stay by Procedure (capped at 15 days)")
ax.set_xlabel("Procedure Code")
ax.set_ylabel("Days")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(PLOT_DIR / "03_procedure_stay.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Access Gap: Patient Migration by Procedure Availability

Do patients in cities without certain procedures have to travel more?

In [ ]:
# For each of the top procedures, identify cities that perform it
top_proc = top_procs[0]  # Largest volume procedure
adopter_cities = set(kidney.loc[kidney["PROC_REA"] == top_proc, "MUNIC_MOV"].unique())
kidney["local_has_top_proc"] = kidney["MUNIC_RES"].astype(str).isin(adopter_cities)

migration_by_access = kidney.groupby("local_has_top_proc").agg(
    migration_rate=("migrated", "mean"),
    avg_stay=("DIAS_PERM", "mean"),
    count=("migrated", "count"),
).round(3)

print(f"Top procedure: {top_proc}")
print(f"Cities performing it: {len(adopter_cities)}")
print(f"\nMigration & stay by local availability:")
print(migration_by_access)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ["No Local Access", "Has Local Access"]
axes[0].bar(labels, migration_by_access["migration_rate"] * 100, color=["#F44336", "#4CAF50"])
axes[0].set_title("Patient Migration Rate")
axes[0].set_ylabel("%")

axes[1].bar(labels, migration_by_access["avg_stay"], color=["#FF9800", "#2196F3"])
axes[1].set_title("Average Length of Stay")
axes[1].set_ylabel("Days")

plt.tight_layout()
plt.savefig(PLOT_DIR / "03_access_gap.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Hospital Comparison: Fast vs Slow Centers

Group hospitals by performance and compare their characteristics to find what makes some faster.

In [ ]:
# Build hospital-level profile
hospital = kidney.groupby("CNES").agg(
    volume=("DIAG_PRINC", "count"),
    avg_stay=("DIAS_PERM", "mean"),
    median_stay=("DIAS_PERM", "median"),
    pct_emergency=("is_emergency", "mean"),
    pct_migrated=("migrated", "mean"),
    mortality=("MORTE", "mean"),
    avg_cost=("VAL_TOT", "mean"),
    city=("MUNIC_MOV", "first"),
).reset_index()

# Add procedure mix per hospital
for proc in top_procs[:4]:
    col_name = f"pct_{proc[-4:]}"
    pct = kidney.groupby("CNES")["PROC_REA"].apply(lambda x: (x == proc).mean())
    hospital = hospital.merge(pct.rename(col_name).reset_index(), on="CNES", how="left")

# Only hospitals with meaningful volume
hospital_sig = hospital[hospital["volume"] >= 50].copy()
print(f"Hospitals with >= 50 kidney stone admissions: {len(hospital_sig)}")

In [ ]:
# Split into fast (below median stay) and slow (above median)
median_stay = hospital_sig["avg_stay"].median()
hospital_sig["speed_group"] = np.where(hospital_sig["avg_stay"] <= median_stay, "Fast", "Slow")

print(f"Median avg stay across hospitals: {median_stay:.1f} days")
print(f"Fast hospitals: {(hospital_sig['speed_group'] == 'Fast').sum()}")
print(f"Slow hospitals: {(hospital_sig['speed_group'] == 'Slow').sum()}")

# Compare characteristics
comparison = hospital_sig.groupby("speed_group").agg(
    n_hospitals=("CNES", "count"),
    avg_volume=("volume", "mean"),
    avg_stay=("avg_stay", "mean"),
    avg_emergency_rate=("pct_emergency", "mean"),
    avg_migration=("pct_migrated", "mean"),
    avg_mortality=("mortality", "mean"),
    avg_cost=("avg_cost", "mean"),
).round(3)

print("\nFast vs Slow hospital comparison:")
print(comparison.T.to_string())

In [ ]:
# Statistical tests: what's significantly different between fast and slow?
test_cols = ["volume", "pct_emergency", "pct_migrated", "mortality", "avg_cost"]
test_cols += [c for c in hospital_sig.columns if c.startswith("pct_") and c not in test_cols]

print("Mann-Whitney U tests (Fast vs Slow):")
print(f"{'Feature':<25} {'U statistic':>12} {'p-value':>10} {'Significant':>12}")
print("-" * 62)

test_results = {}
for col in test_cols:
    if col not in hospital_sig.columns:
        continue
    fast = hospital_sig.loc[hospital_sig["speed_group"] == "Fast", col].dropna()
    slow = hospital_sig.loc[hospital_sig["speed_group"] == "Slow", col].dropna()
    if len(fast) < 5 or len(slow) < 5:
        continue
    stat, pval = stats.mannwhitneyu(fast, slow, alternative="two-sided")
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {col:<25} {stat:>10.0f}   {pval:>8.4f}   {sig:>8}")
    test_results[col] = {"U": float(stat), "p": float(pval), "significant": pval < 0.05}

In [ ]:
# Visualize the key differences
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, title in zip(axes.flat, 
    ["pct_emergency", "volume", "avg_cost", "pct_migrated"],
    ["Emergency Rate", "Volume", "Avg Cost (R$)", "Migration Rate"]):
    for group, color in [("Fast", "#4CAF50"), ("Slow", "#F44336")]:
        data = hospital_sig.loc[hospital_sig["speed_group"] == group, col]
        ax.hist(data, bins=15, alpha=0.6, label=group, color=color, density=True)
    ax.set_title(f"{title}: Fast vs Slow")
    ax.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / "03_fast_vs_slow.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Matched City Comparison

Pick specific city pairs with similar patient volumes but different outcomes.
Compare their hospital profiles to identify operational differences.

In [ ]:
# City-level aggregation
city = kidney.groupby("MUNIC_MOV").agg(
    volume=("DIAG_PRINC", "count"),
    avg_stay=("DIAS_PERM", "mean"),
    median_stay=("DIAS_PERM", "median"),
    pct_emergency=("is_emergency", "mean"),
    pct_migrated=("migrated", "mean"),
    mortality=("MORTE", "mean"),
    n_hospitals=("CNES", "nunique"),
).reset_index()

city = city[city["volume"] >= 200].sort_values("avg_stay")
print(f"Cities with >= 200 admissions: {len(city)}")
print(f"\nTop 10 fastest cities:")
print(city.head(10)[["MUNIC_MOV", "volume", "avg_stay", "pct_emergency", "n_hospitals"]].to_string(index=False))
print(f"\nTop 10 slowest cities:")
print(city.tail(10)[["MUNIC_MOV", "volume", "avg_stay", "pct_emergency", "n_hospitals"]].to_string(index=False))

In [ ]:
# Find matched pairs: similar volume, different outcomes
# For each slow city, find the closest fast city by volume
city["speed"] = np.where(city["avg_stay"] <= city["avg_stay"].median(), "fast", "slow")

fast_cities = city[city["speed"] == "fast"].copy()
slow_cities = city[city["speed"] == "slow"].copy()

pairs = []
used_fast = set()
for _, slow in slow_cities.iterrows():
    candidates = fast_cities[~fast_cities["MUNIC_MOV"].isin(used_fast)].copy()
    if candidates.empty:
        break
    candidates["vol_diff"] = abs(candidates["volume"] - slow["volume"])
    best = candidates.nsmallest(1, "vol_diff").iloc[0]
    if best["vol_diff"] / slow["volume"] < 0.5:  # within 50% volume
        pairs.append({
            "slow_city": slow["MUNIC_MOV"],
            "slow_volume": int(slow["volume"]),
            "slow_stay": round(slow["avg_stay"], 1),
            "slow_er_rate": round(slow["pct_emergency"] * 100, 1),
            "fast_city": best["MUNIC_MOV"],
            "fast_volume": int(best["volume"]),
            "fast_stay": round(best["avg_stay"], 1),
            "fast_er_rate": round(best["pct_emergency"] * 100, 1),
            "stay_gap": round(slow["avg_stay"] - best["avg_stay"], 1),
        })
        used_fast.add(best["MUNIC_MOV"])

pairs_df = pd.DataFrame(pairs).sort_values("stay_gap", ascending=False)
print(f"Found {len(pairs_df)} matched city pairs:")
print(pairs_df.head(10).to_string(index=False))

In [ ]:
# Deep dive into top matched pairs: what hospitals are in each city?
top_pair = pairs_df.iloc[0]

print(f"\n=== Matched Pair Deep Dive ===")
print(f"SLOW: {top_pair['slow_city']} (avg stay {top_pair['slow_stay']}d, ER {top_pair['slow_er_rate']}%)")
print(f"FAST: {top_pair['fast_city']} (avg stay {top_pair['fast_stay']}d, ER {top_pair['fast_er_rate']}%)")
print(f"Gap: {top_pair['stay_gap']} days\n")

for city_code, label in [(top_pair["slow_city"], "SLOW"), (top_pair["fast_city"], "FAST")]:
    city_hospitals = hospital_sig[hospital_sig["city"] == city_code].sort_values("volume", ascending=False)
    print(f"\n--- {label} city: {city_code} ({len(city_hospitals)} hospitals with 50+ admissions) ---")
    if len(city_hospitals) > 0:
        display_cols = ["CNES", "volume", "avg_stay", "pct_emergency"]
        display_cols += [c for c in city_hospitals.columns if c.startswith("pct_") and c != "pct_emergency" and c != "pct_migrated"]
        display_cols = [c for c in display_cols if c in city_hospitals.columns]
        print(city_hospitals[display_cols].head(5).to_string(index=False))

In [ ]:
# Visualize: what differentiates fast and slow cities?
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(city["pct_emergency"] * 100, city["avg_stay"],
                s=city["volume"] / 20, alpha=0.5,
                c=["#4CAF50" if s == "fast" else "#F44336" for s in city["speed"]])
axes[0].set_title("Emergency Rate vs Avg Stay (by city)")
axes[0].set_xlabel("% Emergency Admissions")
axes[0].set_ylabel("Avg LOS (days)")

axes[1].scatter(city["volume"], city["avg_stay"],
                s=30, alpha=0.5,
                c=["#4CAF50" if s == "fast" else "#F44336" for s in city["speed"]])
axes[1].set_title("Volume vs Avg Stay (by city)")
axes[1].set_xlabel("Total Admissions")
axes[1].set_ylabel("Avg LOS (days)")

# Matched pairs visualization
top_n = min(8, len(pairs_df))
pair_subset = pairs_df.head(top_n)
y_pos = range(top_n)

axes[2].barh(y_pos, pair_subset["slow_stay"], height=0.35, label="Slow City", color="#F44336", alpha=0.7)
axes[2].barh([y + 0.35 for y in y_pos], pair_subset["fast_stay"], height=0.35, label="Fast Match", color="#4CAF50", alpha=0.7)
axes[2].set_yticks([y + 0.175 for y in y_pos])
axes[2].set_yticklabels([f"{s} vs {f}" for s, f in zip(pair_subset["slow_city"], pair_subset["fast_city"])], fontsize=7)
axes[2].set_title("Matched City Pairs: LOS Comparison")
axes[2].set_xlabel("Avg LOS (days)")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "03_city_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Procedure Adoption by Hospital Speed

In [ ]:
# For each top procedure, compare adoption rates in fast vs slow hospitals
proc_cols = [c for c in hospital_sig.columns if c.startswith("pct_") and c not in ["pct_emergency", "pct_migrated"]]

if proc_cols:
    fig, axes = plt.subplots(1, len(proc_cols[:4]), figsize=(4 * min(len(proc_cols), 4), 5))
    if len(proc_cols[:4]) == 1:
        axes = [axes]

    for ax, col in zip(axes, proc_cols[:4]):
        for group, color in [("Fast", "#4CAF50"), ("Slow", "#F44336")]:
            data = hospital_sig.loc[hospital_sig["speed_group"] == group, col]
            ax.hist(data, bins=15, alpha=0.6, label=group, color=color, density=True)
        ax.set_title(col)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / "03_procedure_by_speed.png", dpi=150, bbox_inches="tight")
    plt.show()

## 7. Save Results

In [ ]:
hypothesis_metrics = {
    "procedure_decomposition": {
        "total_growth": int(total_growth),
        "top_growth_contributors": {
            str(k): int(v) for k, v in growth_by_proc.head(5).items()
        },
    },
    "access_gap": {
        "top_procedure": str(top_proc),
        "adopter_cities": len(adopter_cities),
        "migration_by_access": migration_by_access.to_dict(),
    },
    "hospital_comparison": {
        "n_hospitals_analyzed": len(hospital_sig),
        "median_stay_cutoff": round(float(median_stay), 1),
        "fast_vs_slow": comparison.to_dict(),
        "statistical_tests": test_results,
    },
    "matched_pairs": {
        "n_pairs": len(pairs_df),
        "avg_stay_gap": round(float(pairs_df["stay_gap"].mean()), 1),
        "max_stay_gap": round(float(pairs_df["stay_gap"].max()), 1),
        "top_pairs": pairs_df.head(5).to_dict(orient="records"),
    },
}

with open(METRICS_DIR / "hypothesis_tests.json", "w") as f:
    json.dump(hypothesis_metrics, f, indent=2, default=str)

print("Hypothesis testing complete.")
print(f"  Procedure growth: top contributor explains {growth_by_proc.iloc[0]/total_growth*100:.0f}% of growth")
print(f"  Hospital comparison: {len(hospital_sig)} hospitals analyzed")
print(f"  Matched pairs: {len(pairs_df)} pairs, avg gap = {pairs_df['stay_gap'].mean():.1f} days")